# Customer Sentiment Intelligence — Full Walkthrough

This notebook demonstrates the complete NLP sentiment pipeline:

1. **Dataset generation & EDA** — synthetic marketing text data
2. **Text preprocessing** — source-aware normalisation
3. **Lexicon baseline** — VADER-inspired rule-based model
4. **Transformer model** — DistilBERT fine-tuning interface
5. **Aspect-based sentiment** — brand / price / creative breakdown
6. **Model evaluation** — F1, calibration, latency, cross-source
7. **Trend detection** — batch reports and alert system

---

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0A0A0A',
    'axes.facecolor': '#0A0A0A',
    'axes.edgecolor': '#2A2A2A',
    'axes.labelcolor': '#CCC',
    'xtick.color': '#888',
    'ytick.color': '#888',
    'grid.color': '#1A1A1A',
    'figure.dpi': 120
})

SENTIMENT_COLORS = {'positive': '#C8F060', 'negative': '#F06090', 'neutral': '#888888'}
EMOTION_COLORS   = {'joy': '#C8F060', 'anger': '#F06090', 'sadness': '#60C8F0',
                    'fear': '#F0A060', 'surprise': '#A060F0', 'neutral': '#555'}
print('✓ Imports ready')

## 1. Dataset Generation & EDA

In [ ]:
from src.data_generator import generate_sentiment_dataset

df = generate_sentiment_dataset(n_samples=2000, seed=42)
print(f'Dataset: {len(df)} samples')
print(f'\nLabel distribution:')
print(df['label'].value_counts())
print(f'\nSource distribution:')
print(df['source'].value_counts())
print(f'\nSample texts:')
for _, row in df.sample(4, random_state=0).iterrows():
    print(f'  [{row["label"].upper():8s}] {row["text"][:80]}...')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Sentiment Dataset Overview', color='#EEE', fontsize=13)

# Label distribution
label_counts = df['label'].value_counts()
axes[0,0].bar(label_counts.index, label_counts.values,
              color=[SENTIMENT_COLORS[l] for l in label_counts.index], alpha=0.85)
for i, (label, count) in enumerate(label_counts.items()):
    axes[0,0].text(i, count + 15, str(count), ha='center', color='#CCC', fontsize=10)
axes[0,0].set_title('Label Distribution', color='#CCC', fontsize=10)
axes[0,0].set_ylabel('Count', color='#CCC')
axes[0,0].grid(True, axis='y')

# Source breakdown by sentiment
source_sent = df.groupby(['source', 'label']).size().unstack(fill_value=0)
sources = source_sent.index
x = np.arange(len(sources))
width = 0.25
for i, label in enumerate(['positive', 'neutral', 'negative']):
    if label in source_sent.columns:
        axes[0,1].bar(x + i*width, source_sent[label], width,
                      label=label, color=SENTIMENT_COLORS[label], alpha=0.8)
axes[0,1].set_xticks(x + width)
axes[0,1].set_xticklabels(sources, color='#CCC')
axes[0,1].set_title('Sentiment by Source', color='#CCC', fontsize=10)
axes[0,1].legend(fontsize=8, framealpha=0.3)
axes[0,1].grid(True, axis='y')

# Text length distribution
for label, color in SENTIMENT_COLORS.items():
    subset = df[df['label'] == label]['char_length']
    axes[1,0].hist(subset, bins=30, alpha=0.5, color=color, label=label, density=True)
axes[1,0].set_title('Text Length Distribution by Sentiment', color='#CCC', fontsize=10)
axes[1,0].set_xlabel('Character Length', color='#CCC')
axes[1,0].legend(fontsize=8, framealpha=0.3)
axes[1,0].grid(True)

# Brand distribution
brand_counts = df['brand'].value_counts()
axes[1,1].barh(brand_counts.index, brand_counts.values, color='#60C8F0', alpha=0.7)
axes[1,1].set_title('Samples per Brand', color='#CCC', fontsize=10)
axes[1,1].set_xlabel('Count', color='#CCC')
axes[1,1].grid(True, axis='x')

plt.tight_layout()
plt.show()

## 2. Text Preprocessing

In [ ]:
from src.sentiment_pipeline import TextPreprocessor

pp = TextPreprocessor()

examples = [
    ('twitter',  '@NovaTech this campaign is sooooo amazing!! 🎉 Check https://example.com #Brilliant #Ad'),
    ('review',   'The product quality is TERRIBLE. Worst brand I have ever purchased from!!!!'),
    ('survey',   'Not sure how I feel about this. The message was okay I guess?'),
    ('social',   '#Zenith just launched a NEW ad 🔥🔥 @zenith_official it\'s actually fire'),
]

print('Text Preprocessing Examples')
print('='*70)
for source, text in examples:
    result = pp.clean(text, source=source)
    print(f'\nSource:   {source}')
    print(f'Original: {text[:70]}')
    print(f'Cleaned:  {result["cleaned_text"][:70]}')
    print(f'Metadata: has_url={result["has_url"]}, emojis={result["emoji_count"]}, hashtags={result["hashtags"]}')

## 3. Lexicon Baseline Analysis

In [ ]:
from src.sentiment_pipeline import LexiconSentimentAnalyser, SentimentIntelligencePipeline

pipeline = SentimentIntelligencePipeline(use_transformer=False)

# Run inference on full dataset
print('Running lexicon pipeline on 2000 samples...')
results = [pipeline.analyse_single(row['text'], source=row['source'])
           for _, row in df.iterrows()]
print(f'✓ Done. Sample result:')
r = results[0]
print(f'  Sentiment: {r.sentiment} ({r.sentiment_confidence:.2f} confidence)')
print(f'  Emotion:   {r.emotion}')
print(f'  Aspects:   {r.aspects}')
print(f'  Latency:   {r.processing_time_ms:.1f}ms')

In [ ]:
# Evaluate predictions vs ground truth
from sklearn.metrics import classification_report, confusion_matrix

true_labels = df['label'].tolist()
pred_labels = [r.sentiment for r in results]
confidences = [r.sentiment_confidence for r in results]

print('Classification Report (Lexicon Baseline)')
print('='*50)
print(classification_report(true_labels, pred_labels, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Lexicon Model Evaluation', color='#EEE', fontsize=13)

# Confusion matrix
labels_order = ['positive', 'neutral', 'negative']
cm = confusion_matrix(true_labels, pred_labels, labels=labels_order)
im = axes[0].imshow(cm, cmap='YlOrRd', aspect='auto')
axes[0].set_xticks(range(3)); axes[0].set_yticks(range(3))
axes[0].set_xticklabels(labels_order, rotation=30, color='#CCC', fontsize=9)
axes[0].set_yticklabels(labels_order, color='#CCC', fontsize=9)
axes[0].set_xlabel('Predicted', color='#CCC'); axes[0].set_ylabel('Actual', color='#CCC')
axes[0].set_title('Confusion Matrix', color='#CCC', fontsize=10)
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, cm[i,j], ha='center', va='center',
                     color='white', fontsize=11, fontweight='bold')

# Confidence distribution
for label, color in SENTIMENT_COLORS.items():
    mask = [p == label for p in pred_labels]
    confs = [c for c, m in zip(confidences, mask) if m]
    if confs:
        axes[1].hist(confs, bins=20, alpha=0.5, color=color, label=label, density=True)
axes[1].set_title('Confidence Distribution by Predicted Label', color='#CCC', fontsize=10)
axes[1].set_xlabel('Confidence Score', color='#CCC')
axes[1].legend(fontsize=8, framealpha=0.3)
axes[1].grid(True)

# Accuracy by source
sources = df['source'].unique()
source_acc = []
for src in sources:
    mask = df['source'] == src
    t = [true_labels[i] for i in df[mask].index]
    p = [pred_labels[i] for i in df[mask].index]
    acc = np.mean([a == b for a, b in zip(t, p)])
    source_acc.append(acc)

bars = axes[2].bar(sources, source_acc, color='#60C8F0', alpha=0.8)
axes[2].axhline(np.mean([a==b for a,b in zip(true_labels,pred_labels)]),
                color='#C8F060', linestyle='--', linewidth=1.5, label='Overall avg')
for bar, val in zip(bars, source_acc):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.2%}', ha='center', color='#CCC', fontsize=9)
axes[2].set_ylim(0, 1)
axes[2].set_title('Accuracy by Source', color='#CCC', fontsize=10)
axes[2].set_ylabel('Accuracy', color='#CCC')
axes[2].legend(fontsize=8, framealpha=0.3)
axes[2].grid(True, axis='y')

plt.tight_layout()
plt.show()

## 4. Aspect-Based Sentiment Breakdown

In [ ]:
# Aggregate aspect sentiments
aspect_data = {}
for r in results:
    for aspect, sentiment in r.aspects.items():
        if aspect not in aspect_data:
            aspect_data[aspect] = Counter()
        aspect_data[aspect][sentiment] += 1

print('Aspect-Based Sentiment Distribution:')
for aspect, counts in sorted(aspect_data.items()):
    total = sum(counts.values())
    print(f'  {aspect:15s}: ', end='')
    for s in ['positive', 'neutral', 'negative']:
        pct = counts.get(s, 0) / total * 100
        print(f'{s}={pct:.0f}%  ', end='')
    print(f'(n={total})')

In [ ]:
aspects = list(aspect_data.keys())
x = np.arange(len(aspects))
width = 0.25

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Aspect-Based Sentiment Analysis', color='#EEE', fontsize=13)

# Stacked bar: sentiment proportions per aspect
bottoms = np.zeros(len(aspects))
for sentiment, color in SENTIMENT_COLORS.items():
    vals = [aspect_data[a].get(sentiment, 0) / sum(aspect_data[a].values()) * 100
            for a in aspects]
    axes[0].bar(aspects, vals, bottom=bottoms, color=color, alpha=0.8, label=sentiment)
    bottoms += np.array(vals)

axes[0].set_ylim(0, 100)
axes[0].set_ylabel('Proportion (%)', color='#CCC')
axes[0].set_title('Sentiment Breakdown by Aspect', color='#CCC', fontsize=10)
axes[0].tick_params(axis='x', colors='#CCC')
axes[0].legend(fontsize=8, framealpha=0.3)
axes[0].grid(True, axis='y')

# Emotion distribution overall
emotion_counts = Counter(r.emotion for r in results)
emotions = list(emotion_counts.keys())
emo_vals = [emotion_counts[e] for e in emotions]
emo_colors = [EMOTION_COLORS.get(e, '#555') for e in emotions]
axes[1].bar(emotions, emo_vals, color=emo_colors, alpha=0.85)
axes[1].set_title('Detected Emotion Distribution', color='#CCC', fontsize=10)
axes[1].set_ylabel('Count', color='#CCC')
axes[1].tick_params(axis='x', colors='#CCC')
axes[1].grid(True, axis='y')

plt.tight_layout()
plt.show()

## 5. Batch Report & Trend Detection

In [ ]:
# Simulate a week of ad comments coming in
weekly_texts = df.sample(100, random_state=1)['text'].tolist()
weekly_sources = df.sample(100, random_state=1)['source'].tolist()

batch_results, report = pipeline.analyse_batch(weekly_texts, weekly_sources)

print('Weekly Batch Report')
print('='*50)
print(f'Documents processed: {report.n_documents}')
print(f'\nSentiment distribution:')
for s, pct in report.sentiment_distribution.items():
    bar = '█' * int(pct / 2)
    print(f'  {s:10s} {bar} {pct:.1f}%')
print(f'\nEmotion distribution:')
for e, pct in sorted(report.emotion_distribution.items(), key=lambda x: -x[1]):
    print(f'  {e:10s}: {pct:.1f}%')
print(f'\nAvg toxicity score: {report.avg_toxicity:.3f}')
print(f'\nTop positive phrases: {report.top_positive_phrases[:5]}')
print(f'Top negative phrases: {report.top_negative_phrases[:5]}')
if report.trend_alerts:
    print(f'\nALERTS:')
    for alert in report.trend_alerts:
        print(f'  {alert}')
else:
    print('\n✓ No trend alerts')

In [ ]:
# Simulate sentiment drift over 12 weeks
weekly_sentiment_history = []
rng = np.random.default_rng(99)

for week in range(12):
    # Introduce a crisis in weeks 7-9
    if 7 <= week <= 9:
        sample = df[df['label'] == 'negative'].sample(
            min(30, len(df[df['label']=='negative'])), random_state=week)
        extra = df[df['label'] != 'negative'].sample(20, random_state=week)
    else:
        sample = df.sample(50, random_state=week)
        extra = pd.DataFrame()
    
    week_texts = pd.concat([sample, extra])['text'].tolist()
    _, week_report = pipeline.analyse_batch(week_texts)
    weekly_sentiment_history.append({
        'week': week + 1,
        **week_report.sentiment_distribution
    })

hist_df = pd.DataFrame(weekly_sentiment_history).fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
fig.suptitle('Sentiment Trend Monitoring — 12 Weeks', color='#EEE', fontsize=13)

for col, color in SENTIMENT_COLORS.items():
    if col in hist_df.columns:
        ax.plot(hist_df['week'], hist_df[col], color=color, linewidth=2,
                marker='o', markersize=5, label=col)

ax.axvspan(7, 9, alpha=0.1, color='#F06090', label='Simulated crisis')
ax.set_xlabel('Week', color='#CCC')
ax.set_ylabel('Sentiment %', color='#CCC')
ax.legend(fontsize=9, framealpha=0.3)
ax.grid(True)

plt.tight_layout()
plt.show()

## 6. DistilBERT Fine-Tuning Interface

The code below shows the fine-tuning interface. Uncomment to run with GPU/CPU.
Requires: `pip install transformers datasets torch`

In [ ]:
from src.sentiment_pipeline import TransformerSentimentAnalyser

# Prepare training data
train_data = df[['text', 'label']].rename(columns={'label': 'label'})
print(f'Training set size: {len(train_data)}')
print(train_data['label'].value_counts())
print()
print('Fine-tuning call (uncomment to run):')
print('''
model = TransformerSentimentAnalyser.fine_tune(
    train_df=train_data,
    text_col="text",
    label_col="label",
    output_dir="models/fine_tuned_sentiment",
    n_epochs=3,
)
# Then load and use:
analyser = TransformerSentimentAnalyser("models/fine_tuned_sentiment")
analyser.load()
result = analyser.predict("This campaign is absolutely brilliant!")
print(result)  # → {"sentiment": "positive", "confidence": 0.97}
''')

## 7. Calibration Analysis

In [ ]:
correct = [p == t for p, t in zip(pred_labels, true_labels)]
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)

bin_acc, bin_conf, bin_size = [], [], []
for i in range(n_bins):
    mask = [(bin_edges[i] <= c < bin_edges[i+1]) for c in confidences]
    if any(mask):
        bin_acc.append(np.mean([correct[j] for j in range(len(correct)) if mask[j]]))
        bin_conf.append(np.mean([confidences[j] for j in range(len(confidences)) if mask[j]]))
        bin_size.append(sum(mask))

ece = sum((s / len(confidences)) * abs(a - c)
          for a, c, s in zip(bin_acc, bin_conf, bin_size))

fig, ax = plt.subplots(figsize=(7, 6))
fig.suptitle(f'Calibration Plot  (ECE = {ece:.3f})', color='#EEE', fontsize=12)

ax.plot([0,1],[0,1], color='#555', linestyle='--', linewidth=1, label='Perfect calibration')
ax.bar(bin_conf, bin_acc, width=0.08, alpha=0.6, color='#60C8F0', label='Model calibration')
ax.plot(bin_conf, bin_acc, color='#C8F060', linewidth=2, marker='o', markersize=6)

ax.set_xlabel('Mean Confidence', color='#CCC')
ax.set_ylabel('Fraction Correct', color='#CCC')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(fontsize=9, framealpha=0.3)
ax.grid(True)
ax.text(0.05, 0.9, f'ECE = {ece:.3f}', color='#F0A060', fontsize=11)

plt.tight_layout()
plt.show()